# Notebook 17: LangGraph - Core Concepts

**🎯 Goal:** To provide a deep understanding of the foundational concepts of LangGraph, the library for building stateful, multi-agent applications. This notebook expands on the introduction from Notebook 9.

## 🧩 Concept Overview

LangGraph is a library that lets you build AI workflows, or 'agents', using a graph-based structure. At its core, it's about defining a **state machine** where each step in the workflow is a node, and the transitions between them are edges.

This approach is powerful because it allows for **cycles**, meaning the workflow can loop back on itself. This is essential for building sophisticated agents that can reflect, revise their work, or ask for clarification.

### 2.1. Directed Acyclic Graphs (DAGs) vs. Cyclic Graphs

Most data processing pipelines (like those built with LangChain Expression Language) are **Directed Acyclic Graphs (DAGs)**. They have a clear start and end, and data flows in one direction.

**LangGraph allows for cycles.** This is the key difference. Why does it matter?
- **Reflection:** An agent can review its own work and decide if it's good enough.
- **Debate:** Multiple agents can go back and forth on a topic.
- **Iteration:** An agent can try a tool, and if it fails, it can try a different one.

```
      +----------+
----->|  Action  |----->
      +----------+
           ^
           | (Loop for revision)
      +----------+
------| Critique |<------
      +----------+
```

### 2.2. State Management

The **state** is the memory of your graph. It's a Python object (usually a `dict` or `TypedDict`) that gets passed between nodes. Each node can read from the state and return an updated version of it.

Using a `TypedDict` is highly recommended for type safety and clarity.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph
import operator

# Using TypedDict to define the shape of our state
class WorkflowState(TypedDict):
    input: str
    # The `operator.add` annotation tells LangGraph to append to this list
    # instead of overwriting it.
    intermediate_steps: Annotated[list, operator.add]
    final_result: str


### 2.3. Nodes

A **node** is a function that performs a unit of work. It takes the current state as input and returns a dictionary containing the updated state values.

A node can be any Python function, and it can call LLMs, tools, APIs, or do any other computation.

In [ ]:
# Example of a simple node
def node_one(state: WorkflowState) -> dict:
    print("--- In Node One ---")
    # This node adds a string to the intermediate_steps list
    return {"intermediate_steps": ["(Node 1) Did some work."]}

# Another example node
def node_two(state: WorkflowState) -> dict:
    print("--- In Node Two ---")
    # This node also adds to the list
    return {"intermediate_steps": ["(Node 2) Did some more work."]}

### 2.4. Edges

**Edges** define the control flow of the graph. They connect the nodes and determine which node runs next.

- **Static Edges:** Always go from one specific node to another. You use `add_edge()` for this.
- **Conditional Edges:** The next node is chosen based on the current state. This is how you implement logic and loops. You use `add_conditional_edges()` for this.

### 2.5. Graph Construction

Putting it all together, you construct the graph using `StateGraph`:

1.  **Instantiate `StateGraph`**: Pass your state definition.
2.  **Add Nodes**: Use `add_node()` to register your functions as nodes.
3.  **Set Entry Point**: Use `set_entry_point()` to tell the graph where to start.
4.  **Add Edges**: Use `add_edge()` or `add_conditional_edges()` to define the flow.
5.  **Compile**: Call `compile()` to create the runnable graph.

In [ ]:
from langgraph.graph import END

# 1. Instantiate the graph
workflow = StateGraph(WorkflowState)

# 2. Add the nodes
workflow.add_node("node_1", node_one)
workflow.add_node("node_2", node_two)

# 3. Set the entry point
workflow.set_entry_point("node_1")

# 4. Add edges
workflow.add_edge("node_1", "node_2")
workflow.add_edge("node_2", END) # END is a special node that terminates the graph

# 5. Compile the graph
app = workflow.compile()

print("Graph compiled successfully!")

In [ ]:
# Let's run it!
initial_state = {"input": "Hello, world!", "intermediate_steps": []}
final_state = app.invoke(initial_state)

print("\n--- Final State ---")
print(final_state)

## 🏁 Summary

You've now learned the fundamental building blocks of LangGraph:

- **State:** The memory of the graph, defined with `TypedDict`.
- **Nodes:** Functions that perform work and update the state.
- **Edges:** The connections that define the workflow's path.
- **`StateGraph`:** The class used to construct and compile your agent.

In the next notebook, we'll explore how to build more dynamic workflows using conditional edges.